In [ ]:
import pandas as pd
import numpy as np
import re
import difflib

In [ ]:
cps = pd.read_csv(r'cellphones_full.csv')
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

In [ ]:
feature_mapping = {
    "Tên": "Name",
    "Giá": "Price",
    "Link": "Link",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
    "Tương thích": "Compatibility",
    "Cảm biến": "Sensors",
}

cps = cps.rename(columns=feature_mapping)


In [ ]:
df = cps.copy()

In [ ]:
antutu = pd.read_csv(r'antutu_score_socket.csv')
att = antutu.copy()

In [ ]:
def extract_refresh_rate(raw_value):
    if pd.isna(raw_value):
        return 0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)\s*hz', s)
    return float(match.group(1)) if match else 0


In [ ]:
df["Refresh Rate"] = df["Display"].apply(extract_refresh_rate)

CLEAN NAME

In [ ]:
def clean_phone_name(raw_name):
    if pd.isna(raw_name):
        return ""

    name = str(raw_name).lower().strip()
    if not name:
        return ""

    # Chuẩn hóa dấu nối và loại bỏ phân đoạn không cần thiết
    name = re.sub(r"[\u2010\u2013\u2014\u2212]", "-", name)
    name = re.sub(r"\s*\|\s*.*$", "", name)
    name = re.sub(r"\bđiện thoại\b", "", name, flags=re.I)
    name = re.sub(r"\b(?:ram|rom)\b", "", name, flags=re.I)

    # Xóa các cụm từ quảng cáo / danh mục không phải model
    patterns_to_delete = [
        r"chính hãng",
        r"vn/?a",
        r"bản quốc tế",
        r"bản chính hãng",
        r"xách tay",
        r"nhập khẩu",
        r"full ?box",
        r"open ?box",
        r"like ?new",
        r"second ?hand",
        r"trả góp",
        r"giá tốt",
        r"giá rẻ",
        r"hàng chính hãng",
        r"hàng.*",
        r"Exynos",
        r"Snapdragon",
        r"special edition",
        r"edition",
        r'china',
        '2021',
        '2022',
        '2023',
        '2024',
        '2025',
        
    ]
    name = re.sub(r"\b(?:" + "|".join(patterns_to_delete) + r")\b", "", name, flags=re.I)

    # Xóa các thông tin mạng và kết nối không phải tên model
    name = re.sub(r"\b(?:4g|5g|nfc|lte|wifi|bluetooth)\b", "", name, flags=re.I)

    # Xóa dung lượng RAM/ROM/ổ cứng
    name = re.sub(r"\b\d+(?:[\.,]\d+)?\s*(?:gb|tb|mb)\b", "", name, flags=re.I)
    name = re.sub(r"\b\d+\s*[x×]\s*\d+\s*(?:gb|tb|mb)\b", "", name, flags=re.I)
    name = re.sub(r"\b\d+\s*[+\/]\s*\d+\s*(?:gb|tb|mb)\b", "", name, flags=re.I)

    # Loại bỏ ký tự không cần và chuẩn hóa khoảng trắng
    name = re.sub(r"[\[\]\(\)\{\}]", " ", name)
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = re.sub(r"\s{2,}", " ", name)
    name = re.sub(r"\b-\b", " ", name)
    name = name.strip()

    return name

In [ ]:
df["Name"] = df["Name"].apply(clean_phone_name)

In [ ]:
def count_matching_names(df1, col1, df2, col2):
    set1 = set(df1[col1].unique())
    set2 = set(df2[col2].unique())
    
    matching = set1 & set2
    
    return {
        'matches': len(matching),
        'df1_unique': len(set1),
        'df2_unique': len(set2),
        'match_rate_df1': f"{(len(matching) / len(set1) * 100):.1f}%",
        'match_rate_df2': f"{(len(matching) / len(set2) * 100):.1f}%",
        'matching_names': sorted(list(matching))
    }

CLEAN PRICE

In [ ]:
def clean_price(raw_price):
    if pd.isna(raw_price):
        return 0
    s = str(raw_price).lower().strip()
    if re.search(r'liên hệ', s):
        return 0
    
    s = re.sub(r'[đ]', '', s)
    s = s.replace('.', '')

    match = re.search(r'(\d+)', s)
    return int(match.group(1)) if match else 0


In [ ]:
df["Price"] = df["Price"].apply(clean_price)

CLEAN STORAGE

In [ ]:
def clean_storage(raw_value):
    if pd.isna(raw_value):
        return 0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)', s)
    if not match:
        return 0
    
    value = float(match.group(1))
    
    if 'tb' in s:
        value = value * 1024
    if 'mb' in s:
        value = value / 1024
    
    return value


In [ ]:
df["RAM"] = df["RAM"].apply(clean_storage)
df["ROM"] = df["ROM"].apply(clean_storage)

CLEAN SIZE, BATTERY

In [ ]:
def clean_metrics(raw_value):
    if pd.isna(raw_value):
        return 0.0

    s = str(raw_value).lower().strip()
    
    match = re.search(r'([\d.]+)', s)
    return float(match.group(1)) if match else 0.0

In [ ]:
cols_to_clean = ["Screen Size","Battery"]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_metrics)

CLEAN CPU


In [ ]:
WORD_CORE_MAP = {
    'dual': 2, 'quad': 4, 'hexa': 6, 'octa': 8,
    'deca': 10, 'nona': 9,
    'tám nhân': 8, 'lõi tám': 8, 'tám lõi': 8,
    'lõi tứ': 4, 'lõi đơn': 1,
}

def normalize(t):
    t = str(t).lower()
    t = re.sub(r'(\d)[,،，](\d)', r'\1.\2', t)
    t = t.replace('，', ',')
    return t

def extract_groups(t):
    patterns = [
        r'(\d+)\s*[x×*]\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×*]\s*[\w\s.\-]+?@\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×*]\s*[\w\s.\-]+?(?:up to|tối đa|lên đến|đến)\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×]\s*[\w\s.\-]+?\((?:tối đa\s*)?(\d+\.?\d*)\s*ghz\)',
        r'(\d+)\s*(?:nhân|lõi)\s+(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×]\s*[a-z]\w+\s+(\d+\.?\d*)\s*ghz',
    ]
    for pat in patterns:
        found = re.findall(pat, t)
        if len(found) >= 2:
            valid = [(int(m[0]), float(m[-1])) for m in found if float(m[-1]) > 0.5]
            if len(valid) >= 2:
                return valid
    return []

def extract_single_freq(t):
    freqs = re.findall(r'(\d+\.?\d*)\s*ghz', t)
    freqs = [float(f) for f in freqs if float(f) > 0.5]
    return max(freqs) if freqs else np.nan

def extract_cpu_features(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return {}
    t = normalize(raw)
    result = {}

    for word, num in WORD_CORE_MAP.items():
        if word in t:
            result['num_cores'] = num
            break
    if 'num_cores' not in result:
        m = re.search(r'(\d+)\s*(nhân|cores?|lõi)', t)
        result['num_cores'] = int(m.group(1)) if m else np.nan

    groups = extract_groups(t)
    if groups:
        groups_sorted = sorted(groups, key=lambda x: x[1], reverse=True)
        result['perf_cores']    = groups_sorted[0][0]
        result['perf_freq_ghz'] = groups_sorted[0][1]
        result['eff_cores']     = groups_sorted[-1][0]
        result['eff_freq_ghz']  = groups_sorted[-1][1]
        # Cộng tổng từ groups nếu num_cores chưa có
        if pd.isna(result.get('num_cores')):
            result['num_cores'] = sum(g[0] for g in groups)
    else:
        single = extract_single_freq(t)
        if not np.isnan(single):
            result['max_freq_ghz'] = single

    if 'perf_cores' not in result:
        m = re.search(r'(\d+)\s*lõi\s*(?:hiệu năng|hiệu suất)', t)
        if m: result['perf_cores'] = int(m.group(1))
    if 'eff_cores' not in result:
        m = re.search(r'(\d+)\s*lõi\s*(?:tiết kiệm|nhỏ)', t)
        if m: result['eff_cores'] = int(m.group(1))

    return result


In [ ]:
features = df['CPU'].apply(extract_cpu_features)
cpu_df   = pd.json_normalize(features)
df       = pd.concat([df.reset_index(drop=True), cpu_df], axis=1)

CLEAN OPERATING SYSTEM

In [ ]:
def advanced_clean_os(text):
    if (
        pd.isna(text)
        or not isinstance(text, str)
        or "cập nhật" in text.lower()
    ):
        return 1, "Android", None

    text = text.strip()
    
    is_android = 1
    os_name = "Android"
    if "ios" in text.lower():
        is_android = 0
        os_name = "iOS"

    # trích xuất số đời (Version)
    os_version = None

    # Trường hợp A: Dòng chỉ chứa mỗi số (Ví dụ: '11')
    if text.isdigit():
        return is_android, os_name, float(text)

    # Trường hợp B: Dòng phức tạp có chữ 'có thể nâng cấp lên Android X'
    if "nâng cấp" in text.lower():
        upgraded_version = re.findall(r"Android\s*(\d+(?:\.\d+)?)", text)
        if upgraded_version:
            # Lấy số phiên bản cuối cùng (cao nhất) trong chuỗi
            return is_android, os_name, float(upgraded_version[-1])

    # Trường hợp C: Dòng thông thường, tìm số đi ngay sau chữ 'Android' hoặc 'iOS'
    version_match = re.search(r"(?:Android|iOS)\s*(\d+(?:\.\d+)?)", text, re.I)
    if version_match:
        os_version = float(version_match.group(1))
    else:
        # Trường hợp như không có số, tạm để None hoặc gán số 8.0/9.0
        os_version = None

    return is_android, os_name, os_version

In [ ]:
df["OS_Is_Android"], df["OS_Name"], \
    df["OS_Version"] = zip(*df["Operating System"].apply(advanced_clean_os))

CLEAN RESOLUTION

In [ ]:
def extract_res_row(text):
    # Nếu dòng bị trống (NaN) hoặc không phải chữ
    if pd.isna(text) or not isinstance(text, str):
        return None, None

    # Tìm cấu trúc a x b ở đầu dòng
    match = re.search(r"^(\d+)\s*[xX×]\s*(\d+)", text.strip())

    if match:
        # Trả về một Tuple gồm (Width, Height) kiểu số nguyên
        return int(match.group(1)), int(match.group(2))

    return None, None

In [ ]:
df["Reso_Width"], df["Reso_Height"] = zip(
    *df["Screen Resolution"].apply(extract_res_row)
)

In [ ]:
def clean_sim_options(text):
    max_nano = 0
    max_esim = 0
    max_micro = 0
    max_mini = 0

    if pd.isna(text) or not isinstance(text, str):
        return max_nano, max_esim, max_micro, max_mini

    text_lower = text.lower()

    options = re.split(r"hoặc|/|;", text_lower)

    for option in options:
        option = option.strip()

        nano_in_opt = 0
        esim_in_opt = 0

        # Xử lý eSIM 
        if "esim" in option:
            match_esim = re.search(r"(\d+)\s*esim", option)
            if match_esim:
                esim_in_opt = int(match_esim.group(1))
            elif "dual" in option or "kép" in option:
                esim_in_opt = 2
            else:
                esim_in_opt = 1

        # Xử lý Nano SIM 
        if "nano" in option or "sim 1 + sim 2" in option:
            match_nano = re.search(r"(\d+)\s*nano", option)
            if match_nano:
                nano_in_opt = int(match_nano.group(1))
            elif (
                "dual" in option
                or "kép" in option
                or "sim 1 + sim 2" in option
            ):
                nano_in_opt = 2
            else:
                nano_in_opt = 1
        elif "2 sim" in option and "nano" in text_lower:
            nano_in_opt = 2
        # Trường hợp ghi mỗi chữ "Nano-SIM" thuần túy
        elif "nano" in option:
            nano_in_opt = 1

        # Cập nhật giá trị max
        max_nano = max(max_nano, nano_in_opt)
        max_esim = max(max_esim, esim_in_opt)

        # Xử lý các loại SIM cổ (Mini, Micro)
        if "micro" in option:
            max_micro = 1
        if "mini" in option:
            max_mini = 1

    return max_nano, max_esim, max_micro, max_mini

In [ ]:
(
    df["Nano_SIM_Count"],
    df["eSIM_Count"],
    df["Micro_SIM_Count"],
    df["Mini_SIM_Count"],
) = zip(*df["SIM Card"].apply(clean_sim_options))

In [ ]:
df.head()

,Name,Price,Link,Screen Size,Display,Rear Camera,Front Camera,Chipset,NFC,ROM,...,max_freq_ghz,OS_Is_Android,OS_Name,OS_Version,Reso_Width,Reso_Height,Nano_SIM_Count,eSIM_Count,Micro_SIM_Count,Mini_SIM_Count
0,iphone 17 pro,34590000,https://cellphones.com.vn/iphone-17-pro.html,6.30,Super Retina XDR,Chính: 48MP khẩu độ ƒ/1.6 OIS hỗ trợ chụp 24MP...,Camera 18MP Center Stage Khẩu độ ƒ/1.9,Chip A19 Pro,Có,256.0,...,NaN,0,iOS,26.0,2622.0,1206.0,2,0,0,0
1,oppo find x9s,23990000,https://cellphones.com.vn/dien-thoai-oppo-find...,6.59,AMOLED,Góc siêu rộng: 50MP; Góc rộng: 50MP; Telephoto...,32MP,MediaTek Dimensity 9500s,Có,256.0,...,NaN,1,Android,NaN,1256.0,2760.0,2,1,0,0
2,iphone 17 pro max,36990000,https://cellphones.com.vn/iphone-17-pro-max.html,6.90,Super Retina XDR,Chính: 48MP khẩu độ ƒ/1.6 OIS hỗ trợ chụp 24MP...,Camera 18MP Center Stage Khẩu độ ƒ/1.9,Chip A19 Pro,Có,256.0,...,NaN,0,iOS,26.0,2868.0,1320.0,2,0,0,0
3,samsung galaxy s26 ultra,30490000,https://cellphones.com.vn/dien-thoai-samsung-g...,6.90,Dynamic AMOLED 2X,Camera siêu rộng: 50MPCamera góc rộng: 200MPCa...,12MP,Snapdragon 8 Elite Gen 5 dành cho Galaxy (3nm),Có,256.0,...,NaN,1,Android,NaN,3120.0,1440.0,2,1,0,0
4,samsung galaxy s26,20490000,https://cellphones.com.vn/dien-thoai-samsung-g...,6.30,Dynamic AMOLED 2X,Camera siêu rộng: 12MPCamera góc rộng: 50MPCam...,12MP,Exynos 2600 (2nm),Có,256.0,...,NaN,1,Android,NaN,2340.0,1080.0,2,1,0,0


In [ ]:
df[['CPU', 'num_cores', 'perf_cores', 'eff_cores', 'perf_freq_ghz', 'eff_freq_ghz', 'max_freq_ghz']]

,CPU,num_cores,perf_cores,eff_cores,perf_freq_ghz,eff_freq_ghz,max_freq_ghz
0,CPU 6 lõi với 2 lõi hiệu năng và 4 lõi tiết ki...,6.0,2.0,4.0,NaN,NaN,NaN
1,8 nhân,8.0,NaN,NaN,NaN,NaN,NaN
2,CPU 6 lõi với 2 lõi hiệu năng và 4 lõi tiết ki...,6.0,2.0,4.0,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
961,2 nhân 2.5 GHz & 6 nhân 2.0 GHz,2.0,2.0,6.0,2.5,2.0,NaN
962,1×Cortex-A710 2.5GHz + 3×Cortex-A710 2.36GHz +...,NaN,NaN,NaN,NaN,NaN,2.5
963,NaN,NaN,NaN,NaN,NaN,NaN,NaN
964,NaN,NaN,NaN,NaN,NaN,NaN,NaN


CLEAN CHIPSET

In [ ]:
_TRASH_VALUES = {
    "",
    "đang cập nhật",
    "mediatek",
    "exynos",
    "snapdragon",
    "bộ xử lý octa-core",
    "asr",
    "asr platform",
    "sc6531e",
    "ums9117",
}
_BRAND_PREFIXES = [
    r"qualcomm\s+(sm|sdm|msm|qm)\w+\s+",   # "Qualcomm SM8350 Snapdragon..." -> "Snapdragon..."
    r"qualcomm\s+",
    r"mediatek\s+",
    r"hisilicon\s+",
    r"samsung\s+",
    r"google\s+",
    r"huawei\s+",
    r"spreadtrum\s+",
    r"unisoc\s+",
    r"apple\s+",
    r"chip\s+",                      
]

def normalize_chipset(text):
    if not isinstance(text, str):
        return ""

    s = text.strip().lower()

    s = re.sub(r"\bthế\s*hệ\b", "gen", s) #"thế hệ" -> "gen"
    s = re.sub(r"\(.*?\)", "", s) #nội dung trong ()
    s = re.sub(r"(\w)\+", r"\1 plus", s) #từ + -> plus
    s = re.sub(r"[®™°•·]", " ", s) #Ký hiệu đặc biệt -> dấu cách
    s = re.sub(r"\b(sm|sdm|msm|apl)\w+\b", "", s) #(sm8350, sdm845, msm8998, apl0698...)

    for pat in _BRAND_PREFIXES:
        s = re.sub(rf"^{pat}", "", s)

    s = re.sub(r"\b(dành cho|cho|danh cho)\s+galaxy\b.*$", "", s)   # "dành cho Galaxy ..."
    s = re.sub(r"\bfor\s+galaxy\b.*$", "", s)                        # "for Galaxy ..."
    s = re.sub(r"\b\d+\s*nhân\b", "", s)                             # "8 nhân", "6 nhân"
    s = re.sub(r"\bocta[\s-]?core\b", "", s)                         # "octa core", "octa-core"
    s = re.sub(r"\b(mobile\s+)?platform\b", "", s)                   # "Mobile Platform"
    s = re.sub(r"\baccelerated\s+edition\b", "", s)                  # "Accelerated Edition"
    s = re.sub(r"\bflagship\b", "", s)                               # "Flagship"
    s = re.sub(r"\btối\s+đa\s+[\d.,]+\s*ghz\b", "", s)              # "tối đa 2.2GHz"
    s = re.sub(r"\btiến\s*trình\b.*$", "", s)                       # "tiến trình 4nm ..."
    s = re.sub(r"\btăng\s+lên\b.*$", "", s)                         # "tăng lên 42% AI ..."
    s = re.sub(r"\b5g\b", "", s)                                     # "5G"
    s = re.sub(r"\b4g\b", "", s)                                     # "4G"

    s = re.sub(r"\b\d+\s*nm\+?\b", "", s) #"6 nm"
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s in _TRASH_VALUES or len(s) < 3:
        return ""

    return s

In [ ]:
def map_chipset_info(df_a, df_c) -> pd.DataFrame:

    map = {}
    for _, row in df_a.iterrows():
        norm = normalize_chipset(row["Chipset"])
        map[norm] = {"antutu_11": row["Antutu_11"], "clock": row["Clock"], "gpu": row["GPU"]}
 
    # Map vào từng dòng
    mapped = df_c["Chipset"].apply(lambda x : map.get(normalize_chipset(x)))

    df_out = df_c.copy()
    df_out["antutu_11"] = mapped.apply(lambda x: x["antutu_11"] if x else None)
    df_out["clock"] = mapped.apply(lambda x: x["clock"] if x else None)
    df_out["gpu"] = mapped.apply(lambda x: x["gpu"] if x else None)
 
    return df_out

In [ ]:
df = map_chipset_info(att, cps)

CLEAN NFC

In [ ]:
df['Công nghệ NFC'] = df['Công nghệ NFC'].map(lambda x : 1 if x == "Có" else 0)

CLEAN CAMERA

In [ ]:
def clean(text):
    # "hỗ trợ chụp 24MP hoặc 48MP"
    text = re.sub(r'hỗ trợ chụp.*?(?=\D{3}|$)', '', text, flags=re.IGNORECASE)
    # "(24MP và 48MP)", "(24MP hoặc 48MP)"
    text = re.sub(r'\([\d.]+\s*MP\s*(?:và|hoặc|or)\s*[\d.]+\s*MP\)', '', text, flags=re.IGNORECASE)
    # "hoặc 48MP" còn sót
    text = re.sub(r'(?:hoặc|hoac|hay|or)\s+[\d.]+\s*MP', '', text, flags=re.IGNORECASE)
    return text

In [ ]:
def extract_mp_values(text):
    text = clean(text)
    vals  = re.findall(r'([\d.]+)\s*(?:MP|megapixel)', text, re.IGNORECASE)
    vals += re.findall(r'([\d.]+)M(?=[^a-zA-Z]|$)', text)
    return [float(v) for v in vals if v.count('.') <= 1 and float(v) >= 0.3]

In [ ]:
def extract_aperture(text):
    vals   = re.findall(r'[fƒ]\s*/?\s*([\d.]+)', text, re.IGNORECASE)
    floats = [float(v) for v in vals if v.count('.') <= 1 and 0.5 <= float(v) <= 6.0]
    return min(floats) if floats else 0

In [ ]:
def count_cameras(text: str, mps: list):
    if len(mps) >= 2:
        return len(mps)
    m = re.search(r'(\d)\s*camera', text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return 1 if mps else 0

In [ ]:
def parse_rear(text):
    if not isinstance(text, str) or not text.strip():
        return {"rear_count": 0, "rear_mp_max": 0, "rear_f/": 0, "rear_ois": 0, "rear_telephoto": 0, "rear_wide": 0}
    mps      = extract_mp_values(text)
    aperture = extract_aperture(text)
    return {
        "rear_count": count_cameras(text, mps),
        "rear_mp_max": max(mps) if mps else 0,
        "rear_f/": aperture if aperture else 0,
        "rear_ois": int(bool(re.search(r'\bOIS\b', text, re.IGNORECASE))),
        "rear_telephoto": int(bool(re.search(r'tele(?:photo)?|zoom quang|kính tiềm vọng|periscope', text, re.IGNORECASE))),
        "rear_wide": int(bool(re.search(r'siêu rộng|ultra.?wide|góc rộng|wide|superwide', text, re.IGNORECASE))),
    }

In [ ]:
def parse_front(text):
    if not isinstance(text, str) or not text.strip():
        return {"front_mp": 0, "front_f/": 0}
    mps = extract_mp_values(text)
    aperture = extract_aperture(text)
    return {
        "front_mp": max(mps) if mps else 0,
        "front_f/": aperture if aperture else 0,
    }

In [ ]:
def parse_camera(df):
    rear  = df["Camera sau"].apply(parse_rear).apply(pd.Series)
    front = df["Camera trước"].apply(parse_front).apply(pd.Series)
    df_out = pd.concat([df, rear, front], axis=1)

    return df_out

In [ ]:
df = parse_camera(df)

In [ ]:
df[["Tên", "rear_count", "rear_mp_max", "rear_f/", "rear_ois", "rear_telephoto", "rear_wide", "front_mp", "front_f/"]].head(10)